# ATLAS — wav2vec2 Transcription

**Model:** `facebook/wav2vec2-large-960h` (CTC-based, pretrained on LibriSpeech 960h)

**What this notebook does:**
1. Loads metadata from Drive (`metadata2/atlas_english_metadata.csv`)
2. Converts WAVs to 16kHz
3. Transcribes singing (control + technique) and speech groups
4. Computes WER + PER
5. Saves all results to Drive for comparison with FireRedASR

**Key difference from FireRedASR:** wav2vec2 is CTC-based, so it cannot hallucinate in the same way as sequence-to-sequence models.

## 0. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Configuration

In [2]:
from pathlib import Path

# ── Google Drive paths ────────────────────────────────────────────────────────
PROJECT_DIR   = Path("/content/drive/MyDrive/atlas")
META_CSV      = PROJECT_DIR / "metadata2" / "atlas_english_metadata.csv"
DRIVE_RESULTS = PROJECT_DIR / "results"

# ── Local paths ───────────────────────────────────────────────────────────────
LOCAL_DIR     = Path("/content/atlas")
EXTRACT_DIR   = LOCAL_DIR / "English_raw"
WAV16K_DIR    = LOCAL_DIR / "wav_16k_wav2vec"
RESULTS_DIR   = LOCAL_DIR / "results_wav2vec"

for d in [LOCAL_DIR, WAV16K_DIR, RESULTS_DIR, DRIVE_RESULTS]:
    d.mkdir(parents=True, exist_ok=True)

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME = "facebook/wav2vec2-large-960h"
BATCH_SIZE = 8

print(f"META_CSV      : {META_CSV}")
print(f"META_CSV exists: {META_CSV.exists()}")
print(f"Extract dir   : {EXTRACT_DIR}")
print(f"WAV16K dir    : {WAV16K_DIR}")
print(f"Model         : {MODEL_NAME}")

META_CSV      : /content/drive/MyDrive/atlas/metadata2/atlas_english_metadata.csv
META_CSV exists: True
Extract dir   : /content/atlas/English_raw
WAV16K dir    : /content/atlas/wav_16k_wav2vec
Model         : facebook/wav2vec2-large-960h


## 2. Install Dependencies

In [3]:
!apt-get install -y -q ffmpeg
!pip install -q transformers torch torchaudio soundfile librosa pandas tqdm jiwer g2p_en datasets

import nltk
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('cmudict')

print("Dependencies installed")

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.3/180.3 kB 15.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 117.7 MB/s eta 0:00:00
Dependencies installed


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package cmudict to /root/nltk_data...
[nltk_data]   Unzipping corpora/cmudict.zip.


## 3. Unzip Dataset (if needed)

In [4]:
import zipfile
from tqdm.notebook import tqdm

ZIP_PATH = PROJECT_DIR / "English.zip"
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

existing_wavs = set(str(p.relative_to(EXTRACT_DIR))
                    for p in EXTRACT_DIR.rglob("*.wav"))
print(f"Already extracted: {len(existing_wavs)} WAVs")

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    missing = [m for m in zf.namelist()
               if not (EXTRACT_DIR / m).exists()]
    print(f"Missing files: {len(missing)}")
    if missing:
        for member in tqdm(missing, desc="Extracting"):
            zf.extract(member, EXTRACT_DIR)
        print("Done!")
    else:
        print("Already complete!")

print(f"Total WAVs: {len(list(EXTRACT_DIR.rglob('*.wav')))}")

Already extracted: 0 WAVs
Missing files: 25984


Extracting:   0%|          | 0/25984 [00:00<?, ?it/s]

Done!
Total WAVs: 6896


## 4. Load Metadata

In [5]:
import pandas as pd
df = pd.read_csv(META_CSV)
print(f"Total utterances : {len(df):,}")
print(f"Groups           : {sorted(df['group'].unique())}")
print(f"Singers          : {sorted(df['singer_id'].unique())}")
print(f"Techniques       : {sorted(df['technique'].unique())}")

# Singing subset (control + technique) with GT lyrics
singing_df = df[
    (df["group"].isin(["control", "technique"])) &
    (df["gt_lyrics"].str.len() > 0)
].copy().reset_index(drop=True)

# Speech subset
speech_df = df[
    (df["group"] == "speech") &
    (df["gt_lyrics"].str.len() > 0)
].copy().reset_index(drop=True)

print(f"\nSinging utterances: {len(singing_df):,}")
print(f"Speech utterances : {len(speech_df):,}")

Total utterances : 6,892
Groups           : ['control', 'speech', 'technique']
Singers          : ['EN-Alto-1', 'EN-Alto-2', 'EN-Tenor-1']
Techniques       : ['breathy', 'glissando', 'mixed_falsetto', 'pharyngeal', 'vibrato']

Singing utterances: 3,646
Speech utterances : 1,563


## 5. Convert WAVs to 16kHz

In [6]:
import subprocess
from pathlib import Path

def convert_to_16k(src: Path, dst: Path) -> bool:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return True
    result = subprocess.run(
        ["ffmpeg", "-y", "-i", str(src),
         "-ar", "16000", "-ac", "1",
         "-acodec", "pcm_s16le", str(dst)],
        capture_output=True
    )
    return result.returncode == 0

def convert_df(input_df, prefix=""):
    wav16k_paths, failed = [], []
    for _, row in tqdm(input_df.iterrows(), total=len(input_df), desc=f"Converting {prefix}"):
        src = Path(row["wav_path"])
        dst = WAV16K_DIR / (prefix + src.parent.name + "__" + src.name)
        if convert_to_16k(src, dst):
            wav16k_paths.append(str(dst))
        else:
            wav16k_paths.append(None)
            failed.append(src.name)
    print(f"Converted: {len(input_df) - len(failed):,} / {len(input_df):,}")
    if failed:
        print(f"Failed: {len(failed)} — {failed[:3]}")
    return wav16k_paths

singing_df["wav_16k"] = convert_df(singing_df, prefix="singing__")
speech_df["wav_16k"]  = convert_df(speech_df,  prefix="speech__")

Converting singing__:   0%|          | 0/3646 [00:00<?, ?it/s]

Converted: 3,646 / 3,646


Converting speech__:   0%|          | 0/1563 [00:00<?, ?it/s]

Converted: 1,563 / 1,563


## 6. Load wav2vec2 Model

In [7]:
import torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

print(f"Loading {MODEL_NAME} ...")
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_NAME).to(device)
model.eval()
print("Model loaded")

Device: cuda
Loading facebook/wav2vec2-large-960h ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/404 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-large-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Model loaded


## 7. Transcription Function

In [8]:
import torch
import soundfile as sf
import numpy as np
from tqdm.notebook import tqdm

def transcribe_batch(wav_paths: list) -> list:
    """Transcribe a batch of 16kHz WAV files using wav2vec2."""
    audios = []
    for path in wav_paths:
        audio, sr = sf.read(path)
        if sr != 16000:
            import librosa
            audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
        audios.append(audio.astype(np.float32))

    inputs = processor(
        audios,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )
    input_values = inputs.input_values.to(device)
    attention_mask = inputs.get("attention_mask", None)
    if attention_mask is not None:
        attention_mask = attention_mask.to(device)

    with torch.no_grad():
        logits = model(input_values, attention_mask=attention_mask).logits

    predicted_ids = torch.argmax(logits, dim=-1)
    transcriptions = processor.batch_decode(predicted_ids)
    return [t.lower().strip() for t in transcriptions]


def transcribe_df(input_df, desc="Transcribing"):
    """Transcribe all utterances in a DataFrame."""
    valid_df = input_df[input_df["wav_16k"].notna()].copy().reset_index(drop=True)
    print(f"{desc}: {len(valid_df):,} utterances in batches of {BATCH_SIZE}")

    all_transcriptions = []
    indices = list(range(len(valid_df)))
    batches = [indices[i:i+BATCH_SIZE] for i in range(0, len(indices), BATCH_SIZE)]

    for batch_idx in tqdm(batches, desc=desc):
        batch_rows = valid_df.iloc[batch_idx]
        wavs = [row["wav_16k"] for _, row in batch_rows.iterrows()]
        try:
            results = transcribe_batch(wavs)
            all_transcriptions.extend(results)
        except Exception as e:
            print(f"Batch error: {e}")
            all_transcriptions.extend([""] * len(batch_idx))

    valid_df["asr_hypothesis"] = all_transcriptions
    return valid_df

print("Transcription function ready")

Transcription function ready


## 8. Transcribe Singing

In [9]:
valid_singing_df = transcribe_df(singing_df, desc="Transcribing Singing")
print(f"\nDone — {len(valid_singing_df):,} utterances")
print(valid_singing_df[["singer_id", "technique", "group", "gt_lyrics", "asr_hypothesis"]].head(5).to_string())

Transcribing Singing: 3,646 utterances in batches of 8


Transcribing Singing:   0%|          | 0/456 [00:00<?, ?it/s]


Done — 3,646 utterances
    singer_id       technique      group                                                                    gt_lyrics                                                          asr_hypothesis
0  EN-Tenor-1  mixed_falsetto  technique  i know you will i know you will i know that you will will you still love me  i know you well i know you well i know that you w well you still on me
1  EN-Tenor-1  mixed_falsetto  technique                 will you still love me when i got nothing but my aching soul                well you still lone me wen i got nothing ba my egging so
2  EN-Tenor-1  mixed_falsetto  technique                                                 when i'm no longer beautiful                                           o wen i'm no longer beo da fo
3  EN-Tenor-1  mixed_falsetto  technique                will you still love me when i'm no longer young and beautiful               well you still lone me when i'm a longer yarn an beu a fo
4  EN-Tenor-1  mixed_fals

In [10]:
# Save RAW immediately
raw_singing = DRIVE_RESULTS / "final_wav2vec2_large_960h__singing_transcriptions_RAW.csv"
valid_singing_df[["clean_name", "singer_id", "technique", "group",
                  "gt_lyrics", "asr_hypothesis"]].to_csv(raw_singing, index=False)
print(f"Saved {raw_singing}")

Saved /content/drive/MyDrive/atlas/results/final_wav2vec2_large_960h__singing_transcriptions_RAW.csv


## 9. Transcribe Speech

In [11]:
valid_speech_df = transcribe_df(speech_df, desc="Transcribing Speech")
print(f"\nDone — {len(valid_speech_df):,} utterances")
print(valid_speech_df[["singer_id", "technique", "group", "gt_lyrics", "asr_hypothesis"]].head(5).to_string())

Transcribing Speech: 1,563 utterances in batches of 8


Transcribing Speech:   0%|          | 0/196 [00:00<?, ?it/s]


Done — 1,563 utterances
    singer_id       technique   group                                                                    gt_lyrics                                                                asr_hypothesis
0  EN-Tenor-1  mixed_falsetto  speech  i know you will i know you will i know that you will will you still love me  i know you will i know you will i know that you will while you still love me
1  EN-Tenor-1  mixed_falsetto  speech                 will you still love me when i got nothing but my aching soul                 where you still love me when i got nothing but my aching soul
2  EN-Tenor-1  mixed_falsetto  speech                                                 when i'm no longer beautiful                                                     winam no longer beautiful
3  EN-Tenor-1  mixed_falsetto  speech                will you still love me when i'm no longer young and beautiful                where you still love me when i'm no longer young and beautiful
4  EN-Teno

In [12]:
# Save RAW immediately
raw_speech = DRIVE_RESULTS / "final_wav2vec2_large_960h__speech_transcriptions_RAW.csv"
valid_speech_df[["clean_name", "singer_id", "technique", "group",
                 "gt_lyrics", "asr_hypothesis"]].to_csv(raw_speech, index=False)
print(f"Saved {raw_speech}")

Saved /content/drive/MyDrive/atlas/results/final_wav2vec2_large_960h__speech_transcriptions_RAW.csv


## 10. Compute WER + PER

In [13]:
import re
from jiwer import wer
from g2p_en import G2p

g2p = G2p()

def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"[^a-z ]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def text_to_phonemes(text: str) -> str:
    text = clean_text(text)
    if not text:
        return ""
    phonemes = g2p(text)
    return " ".join([p for p in phonemes if p.strip() and p != " "])

def compute_wer_per(input_df):
    wer_list, per_list = [], []
    for _, row in input_df.iterrows():
        gt  = clean_text(str(row["gt_lyrics"]))
        hyp = clean_text(str(row["asr_hypothesis"]))
        wer_list.append(wer(gt, hyp) if gt else None)
        gt_ph  = text_to_phonemes(gt)
        hyp_ph = text_to_phonemes(hyp)
        per_list.append(wer(gt_ph, hyp_ph) if gt_ph else None)
    input_df["wer"] = wer_list
    input_df["per"] = per_list
    return input_df

print("Computing WER/PER for singing...")
valid_singing_df = compute_wer_per(valid_singing_df)
print(f"Singing — Overall WER: {valid_singing_df['wer'].mean():.1%}  PER: {valid_singing_df['per'].mean():.1%}")

print("\nComputing WER/PER for speech...")
valid_speech_df = compute_wer_per(valid_speech_df)
print(f"Speech  — Overall WER: {valid_speech_df['wer'].mean():.1%}  PER: {valid_speech_df['per'].mean():.1%}")

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


Computing WER/PER for singing...
Singing — Overall WER: 123.5%  PER: 104.6%

Computing WER/PER for speech...
Speech  — Overall WER: 120.7%  PER: 105.6%


In [15]:
for name, df_ in [("singing", valid_singing_df), ("speech", valid_speech_df)]:
    halluc_rate = (df_["wer"] > 1.0).mean()
    no_halluc   = df_[df_["wer"] < 1.0]
    print(f"{name}:")
    print(f"  Hallucination rate  : {halluc_rate:.1%}")
    print(f"  WER (no halluc)     : {no_halluc['wer'].mean():.1%}")
    print(f"  PER (no halluc)     : {no_halluc['per'].mean():.1%}")
    print()

singing:
  Hallucination rate  : 38.2%
  WER (no halluc)     : 88.3%
  PER (no halluc)     : 78.7%

speech:
  Hallucination rate  : 34.8%
  WER (no halluc)     : 87.6%
  PER (no halluc)     : 79.9%



In [16]:
print("WER / PER by Technique:")
print(valid_singing_df.groupby("technique")[["wer","per"]].mean().round(3).sort_values("wer", ascending=False).to_string())
print()
print("WER / PER by Singer:")
print(valid_singing_df.groupby("singer_id")[["wer","per"]].mean().round(3).to_string())
print()
print("WER / PER by Group:")
print(valid_singing_df.groupby("group")[["wer","per"]].mean().round(3).to_string())
print()
halluc = (valid_singing_df["wer"] > 1.0).mean()
no_halluc = valid_singing_df[valid_singing_df["wer"] < 1.0]
print(f"Hallucination rate : {halluc:.1%}")
print(f"WER (no halluc)    : {no_halluc['wer'].mean():.1%}")
print(f"PER (no halluc)    : {no_halluc['per'].mean():.1%}")

WER / PER by Technique:
                  wer    per
technique                   
vibrato         1.349  1.140
breathy         1.325  1.154
glissando       1.265  1.074
pharyngeal      1.243  1.092
mixed_falsetto  1.156  0.952

WER / PER by Singer:
              wer    per
singer_id               
EN-Alto-1   1.211  1.009
EN-Alto-2   1.239  1.053
EN-Tenor-1  1.244  1.059

WER / PER by Group:
             wer    per
group                  
control    1.228  1.038
technique  1.240  1.052

Hallucination rate : 38.2%
WER (no halluc)    : 88.3%
PER (no halluc)    : 78.7%


In [17]:
wer_list, per_list = [], []

for _, row in valid_speech_df.iterrows():
    gt  = clean_text(str(row["gt_lyrics"]))
    hyp = clean_text(str(row["asr_hypothesis"]))
    wer_list.append(wer(gt, hyp) if gt else None)
    gt_ph  = text_to_phonemes(gt)
    hyp_ph = text_to_phonemes(hyp)
    per_list.append(wer(gt_ph, hyp_ph) if gt_ph else None)

valid_speech_df["wer"] = wer_list
valid_speech_df["per"] = per_list

print(f"Overall WER: {valid_speech_df['wer'].mean():.1%}")
print(f"Overall PER: {valid_speech_df['per'].mean():.1%}")
print()
print(valid_speech_df.groupby("singer_id")[["wer", "per"]].mean().round(3))
print()
print(valid_speech_df.groupby("technique")[["wer", "per"]].mean().round(3))

halluc = (valid_speech_df["wer"] > 1.0).mean()
no_halluc = valid_speech_df[valid_speech_df["wer"] < 1.0]
print(f"Hallucination rate : {halluc:.1%}")
print(f"WER (no halluc)    : {no_halluc['wer'].mean():.1%}")
print(f"PER (no halluc)    : {no_halluc['per'].mean():.1%}")

Overall WER: 120.7%
Overall PER: 105.6%

              wer    per
singer_id               
EN-Alto-1   1.183  1.015
EN-Alto-2   1.210  1.067
EN-Tenor-1  1.217  1.071

                  wer    per
technique                   
breathy         1.236  1.091
glissando       1.215  1.076
mixed_falsetto  1.119  0.966
pharyngeal      1.223  1.080
vibrato         1.343  1.168
Hallucination rate : 34.8%
WER (no halluc)    : 87.6%
PER (no halluc)    : 79.9%


In [18]:
empty_hyp = (valid_singing_df["asr_hypothesis"].str.strip().str.len() == 0).sum()
print(f"Empty hypotheses: {empty_hyp} / {len(valid_singing_df)}")
print(valid_singing_df[valid_singing_df["wer"] > 1.0][["gt_lyrics", "asr_hypothesis"]].head(5).to_string())

Empty hypotheses: 7 / 3646
                                                      gt_lyrics                                                          asr_hypothesis
5                            diamonds brilliant and bel air now                                            dy a mn's brelyant am ber no
25      i'm falling in all the good times i find myself longing  i know you well i know you well i know that you w well you still on me
26     or do you need more ain't it hard keeping it so hardcore                well you still lone me wen i got nothing ba my egging so
28  tell me something boy aren't you tired tryna fill that void               well you still lone me when i'm a longer yarn an beu a fo
29                for change and in the bad times i fear myself        the crazy days see the igt the way you play with me like a child


## 11. Save WER/PER Results

In [19]:
# Save full results
singing_csv = DRIVE_RESULTS / "final_wav2vec2_large_960h__singing_WER_PER.csv"
valid_singing_df[["clean_name", "singer_id", "technique", "group",
                  "gt_lyrics", "asr_hypothesis", "wer", "per"]].to_csv(singing_csv, index=False)
print(f"Saved → {singing_csv}")

speech_csv = DRIVE_RESULTS / "final_wav2vec2_large_960h__speech_WER_PER.csv"
valid_speech_df[["clean_name", "singer_id", "technique", "group",
                 "gt_lyrics", "asr_hypothesis", "wer", "per"]].to_csv(speech_csv, index=False)
print(f"Saved → {speech_csv}")

Saved → /content/drive/MyDrive/atlas/results/final_wav2vec2_large_960h__singing_WER_PER.csv
Saved → /content/drive/MyDrive/atlas/results/final_wav2vec2_large_960h__speech_WER_PER.csv


## sanity check with LibriSpeech

In [20]:
# Sanity Check — wav2vec2 on LibriSpeech test-clean
from datasets import load_dataset
import soundfile as sf

print("Loading LibriSpeech test-clean (streaming)...")
libri = load_dataset("openslr/librispeech_asr", "clean", split="test", streaming=True)

libri_samples = []
for i, sample in enumerate(libri):
    if i >= 1563:
        break
    libri_samples.append(sample)

print(f"Loaded {len(libri_samples)} utterances")

# Save WAVs
LIBRI_DIR = LOCAL_DIR / "librispeech_16k"
LIBRI_DIR.mkdir(exist_ok=True)
libri_records = []

for i, sample in enumerate(tqdm(libri_samples, desc="Saving WAVs")):
    wav_path = LIBRI_DIR / f"libri_{i:05d}.wav"
    if not wav_path.exists():
        sf.write(str(wav_path), sample["audio"]["array"],
                 sample["audio"]["sampling_rate"])
    libri_records.append({
        "uttid":    f"libri_{i:05d}",
        "wav_path": str(wav_path),
        "gt_text":  sample["text"].lower()
    })

libri_df = pd.DataFrame(libri_records)

# Transcribe
libri_transcriptions = []
batches = [list(range(len(libri_df)))[i:i+BATCH_SIZE]
           for i in range(0, len(libri_df), BATCH_SIZE)]

for batch_idx in tqdm(batches, desc="Transcribing LibriSpeech"):
    batch_rows = libri_df.iloc[batch_idx]
    wavs = [row["wav_path"] for _, row in batch_rows.iterrows()]
    try:
        results = transcribe_batch(wavs)
        libri_transcriptions.extend(results)
    except Exception as e:
        print(f"Batch error: {e}")
        libri_transcriptions.extend([""] * len(batch_idx))

libri_df["asr_hypothesis"] = libri_transcriptions
libri_df[["uttid", "gt_text", "asr_hypothesis"]].to_csv(
    DRIVE_RESULTS / "sanity_check_wav2vec2_librispeech_RAW.csv", index=False)
print("Saved RAW!")

# WER only
wer_list = []
for _, row in tqdm(libri_df.iterrows(), total=len(libri_df), desc="Computing WER"):
    gt  = clean_text(str(row["gt_text"]))
    hyp = clean_text(str(row["asr_hypothesis"]))
    wer_list.append(wer(gt, hyp) if gt else None)

libri_df["wer"] = wer_list
halluc = (libri_df["wer"] > 1.0).mean()

print(f"\nLibriSpeech WER    : {libri_df['wer'].mean():.1%}")
print(f"Hallucination rate : {halluc:.1%}")

libri_df[["uttid", "gt_text", "asr_hypothesis", "wer"]].to_csv(
    DRIVE_RESULTS / "sanity_check_wav2vec2_librispeech_WER.csv", index=False)
print("Saved!")

Loading LibriSpeech test-clean (streaming)...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Loaded 1563 utterances


Saving WAVs:   0%|          | 0/1563 [00:00<?, ?it/s]

Transcribing LibriSpeech:   0%|          | 0/196 [00:00<?, ?it/s]

Saved RAW!


Computing WER:   0%|          | 0/1563 [00:00<?, ?it/s]


LibriSpeech WER    : 4.3%
Hallucination rate : 0.0%
Saved!


In [21]:
empty_hyp = (libri_df["asr_hypothesis"].str.strip().str.len() == 0).sum()
print(f"Empty hypotheses: {empty_hyp} / {len(libri_df)}")
print()

# Hallucinated
halluc_df = libri_df[libri_df["wer"] > 1.0]
print(f"Hallucinated: {len(halluc_df)}")
if len(halluc_df) > 0:
    print(halluc_df[["gt_text", "asr_hypothesis"]].head(5).to_string())
print()

# Worst WER samples
print("=== Worst WER samples ===")
print(libri_df.nlargest(5, "wer")[["gt_text", "asr_hypothesis", "wer"]].to_string())

Empty hypotheses: 2 / 1563

Hallucinated: 0

=== Worst WER samples ===
                      gt_text asr_hypothesis  wer
349                 direction       directon  1.0
430          explain yourself    xplain yslf  1.0
438                 indeed ah                 1.0
1069                verse two       virst to  1.0
1401  by reason and affection    snan afeton  1.0


In [22]:
import numpy as np

#mean number of words per utterance
libri_words = libri_df["gt_text"].str.split().str.len()
singing_words = valid_singing_df["gt_lyrics"].str.split().str.len()

print(f"LibriSpeech mean words/utterance : {libri_words.mean():.1f}")
print(f"GTSinger mean words/utterance    : {singing_words.mean():.1f}")
print(f"LibriSpeech max words            : {libri_words.max()}")
print(f"GTSinger max words               : {singing_words.max()}")

LibriSpeech mean words/utterance : 20.4
GTSinger mean words/utterance    : 13.0
LibriSpeech max words            : 96
GTSinger max words               : 46
